### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="kdd_cup_09_appetency",
    dataset_year="2008",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://kdd.org/kdd-cup/view/kdd-cup-2009/Data",
    download_description="""
We download the data and the labels from the KDD challenge website:

mkdir -p local-data-warehouse/kdd_cup_09_appetency && \
wget -P local-data-warehouse/kdd_cup_09_appetency https://kdd.org/cupfiles/KDDCupData/2009/orange_small_train.data.zip && \
unzip local-data-warehouse/kdd_cup_09_appetency/orange_small_train.data.zip -d local-data-warehouse/kdd_cup_09_appetency/ && \
rm local-data-warehouse/kdd_cup_09_appetency/orange_small_train.data.zip && \
wget -P local-data-warehouse/kdd_cup_09_appetency https://kdd.org/cupfiles/KDDCupData/2009/orange_small_train_appetency.labels
""",
    # References
    academic_reference_bibtex=r"""@inproceedings{guyon2009analysis,
  title={Analysis of the kdd cup 2009: Fast scoring on a large orange customer database},
  author={Guyon, Isabelle and Lemaire, Vincent and Boull{\'e}, Marc and Dror, Gideon and Vogel, David},
  booktitle={KDD-Cup 2009 Competition},
  pages={1--22},
  year={2009},
  organization={PMLR}
}
""",
    academic_reference_bibtex_key="guyon2009analysis",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We use the small training data from the original data (230 features).
- We use appetency as label.
- We dropped empty columns: "Var8", "Var15", "Var20", "Var31", "Var32", "Var39", "Var42", "Var48", "Var52", "Var55", "Var79", "Var141", "Var167", "Var169", "Var175", "Var185", "Var209", "Var230".
- Anomaly: the feature names and categorical feature values have no semantic meaning.
- Anomaly: this dataset has many missing values.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Appetency",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Appetency",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "orange_small_train.data", sep="\t")
df_y = pd.read_csv(dataset_mold.path / "orange_small_train_appetency.labels", header=None)

target_feature = "Appetency"
df[target_feature] = df_y[0].astype("category")

empty_cols = [
    "Var8","Var15","Var20","Var31","Var32","Var39","Var42","Var48","Var52",
    "Var55","Var79","Var141","Var167","Var169","Var175","Var185","Var209","Var230",
]
df.drop(columns=empty_cols, inplace=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

cat_cols = [
    "Var192","Var193","Var194","Var195","Var196","Var197","Var198","Var199","Var200",
    "Var201","Var202","Var203","Var204","Var205","Var206","Var207","Var208","Var210",
    "Var211","Var212","Var214","Var216","Var217","Var218","Var219","Var220","Var222",
    "Var223","Var225","Var226","Var227","Var228","Var229",
    'Var191', 'Var213', 'Var215', 'Var221', 'Var224'
]
df[cat_cols] = df[cat_cols].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()